In [ ]:
#import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
#load dataset
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train = pd.read_csv('/kaggle/input/datasets/yasserh/titanic-dataset/Titanic-Dataset.csv')
print(train.head())
print(train.shape)

In [ ]:
#Preprocessing
train['Age'] = train['Age'].fillna(
    train['Age'].median()
)

train['Embarked'] = train['Embarked'].fillna(
    train['Embarked'].mode()[0]
)

In [ ]:
#Feature Enginnering
train['FamilySize'] = (
    train['SibSp']
    + train['Parch']
    + 1
)

train['IsAlone'] = (
    train['FamilySize'] == 1
).astype(int)

In [ ]:
#Encode
le = LabelEncoder()

train['Sex'] = le.fit_transform(train['Sex'])
train['Embarked'] = le.fit_transform(train['Embarked'])

In [ ]:
#drop unwanted column
train.drop(
    ['PassengerId', 'Name', 'Ticket', 'Cabin'],
    axis=1,
    inplace=True,
    errors='ignore'
)

In [ ]:
#create X and Y
X = train.drop('Survived', axis=1)
y = train['Survived']

print(X.shape)
print(y.shape)

In [ ]:
#Train Test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)

In [ ]:
#compare models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)

    print(name, ":", acc)

In [ ]:
#Evaluate best model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
#confusion model
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d')

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
#classification report
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
#Feature importance
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance)

In [ ]:
#Visualization
plt.figure(figsize=(8,5))

sns.barplot(
    x='Importance',
    y='Feature',
    data=importance
)

plt.title("Feature Importance")
plt.show()

In [ ]:
#EDA
#Survival distribution
sns.countplot(x='Survived', data=train)
plt.title("Survival Distribution")
plt.show()

In [ ]:
#survival by gender
sns.countplot(
    x='Sex',
    hue='Survived',
    data=train
)

plt.title("Survival by Gender")
plt.show()

In [ ]:
#survival by passenger class
sns.countplot(
    x='Pclass',
    hue='Survived',
    data=train
)

plt.title("Survival by Passenger Class")
plt.show()

In [ ]:
#age distrubution
plt.figure(figsize=(8,5))

sns.histplot(
    train['Age'],
    bins=20
)

plt.title("Age Distribution")
plt.show()

In [ ]:
#save model
import joblib

joblib.dump(
    rf,
    'titanic_survival_model.pkl'
)

print("Model Saved Successfully")